# ProyectoG9 - Proceso ETL, Modelo Estrella y Tabla de Hechos

## Inteligencia de Negocios - Componente Práctico Semana 3

### Grupo 9

**Integrantes:**

- Guevara Paz y Miño Juan Diego
- Moreno Jeremy
- Solis Paulo


## 1. Introducción del proyecto

El presente notebook corresponde al desarrollo del proyecto práctico de la Semana 3, enfocado en la construcción de un modelo estrella a partir de datos previamente limpiados y transformados en el proceso ETL.

El proyecto trabaja con información relacionada con **ciberseguridad global e indicadores digitales por país**. Para ello, se utilizan tres fuentes principales:

1. **Global Cybersecurity Threats 2015-2024:** contiene información sobre incidentes globales de ciberseguridad, incluyendo país, año, tipo de ataque, industria afectada, pérdida financiera, usuarios afectados, fuente del ataque, vulnerabilidad, mecanismo de defensa y tiempo de resolución.
2. **Cyber Security Index:** contiene indicadores de ciberseguridad por país, como CEI, GCI, NCSI y DDL.
3. **Internet Users by Country:** contiene información sobre usuarios de internet, población y porcentaje de penetración de internet por país.

El objetivo de esta etapa es preparar los datos para una estructura analítica tipo Data Warehouse mediante:

- carga de DataFrames transformados,
- creación del modelo estrella,
- creación de dimensiones,
- creación de una tabla de hechos.

Este modelo permitirá analizar los incidentes de ciberseguridad desde diferentes perspectivas, como país, año, tipo de ataque, industria afectada y nivel de impacto.


## 2. Importación de librerías

En esta sección se importan las librerías necesarias para cargar, limpiar, transformar y estructurar los datos.

Se utiliza principalmente:

- `pandas` para manipulación de DataFrames.
- `pathlib` para el manejo ordenado de rutas.
- `re` y `unicodedata` para normalización de textos.
- `IPython.display` para presentar tablas dentro del notebook.


In [1]:
import pandas as pd
from pathlib import Path
from IPython.display import display
import re
import unicodedata

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Librerías importadas correctamente.")

Librerías importadas correctamente.


## 3. Carga de DataFrames transformados

En esta sección se cargan las fuentes originales del proyecto y se aplican nuevamente las transformaciones trabajadas en la Semana 2.

Esto permite obtener DataFrames limpios y preparados para construir el modelo estrella sin modificar los archivos originales.

Los DataFrames principales generados en esta sección son:

- `df_threats_transformed`
- `df_cyber_security_transformed`
- `df_internet_users_transformed`
- `df_threats_enriched`


In [2]:
# Configuración de rutas
DATA_DIR = Path("data")

ruta_threats = DATA_DIR / "Global_Cybersecurity_Threats_2015-2024.csv"
ruta_cyber_security = DATA_DIR / "Cyber_security.csv"
ruta_internet_users = DATA_DIR / "internet_users_by_country_cleaned.csv"

rutas_requeridas = [ruta_threats, ruta_cyber_security, ruta_internet_users]
archivos_faltantes = [str(ruta) for ruta in rutas_requeridas if not ruta.exists()]

if archivos_faltantes:
    raise FileNotFoundError(
        "No se encontraron los siguientes archivos requeridos: "
        + ", ".join(archivos_faltantes)
    )

# Carga de fuentes originales
df_threats = pd.read_csv(ruta_threats)
df_cyber_security = pd.read_csv(ruta_cyber_security)
df_internet_users = pd.read_csv(ruta_internet_users)

datasets = {
    "Global Cybersecurity Threats": df_threats,
    "Cyber Security Index": df_cyber_security,
    "Internet Users by Country": df_internet_users
}

resumen_carga = pd.DataFrame([
    {
        "fuente": nombre,
        "filas": df.shape[0],
        "columnas": df.shape[1],
        "valores_nulos": int(df.isnull().sum().sum()),
        "registros_duplicados": int(df.duplicated().sum())
    }
    for nombre, df in datasets.items()
])

resumen_carga

,fuente,filas,columnas,valores_nulos,registros_duplicados
0,Global Cybersecurity Threats,3000,10,0,0
1,Cyber Security Index,192,6,151,0
2,Internet Users by Country,215,7,0,0


### 3.1 Funciones de limpieza y normalización

Estas funciones permiten normalizar columnas, limpiar textos, homologar nombres de países y convertir columnas numéricas.


In [3]:
COUNTRY_MAPPING = {
    "USA": "United States",
    "U.S.A.": "United States",
    "US": "United States",
    "UK": "United Kingdom",
    "U.K.": "United Kingdom"
}


def normalizar_nombre_columna(columna):
    columna = columna.strip().lower()
    columna = columna.replace("$", "")
    columna = re.sub(r"[()/%]+", " ", columna)
    columna = re.sub(r"[^a-z0-9]+", "_", columna)
    columna = re.sub(r"_+", "_", columna)
    columna = columna.strip("_")
    return columna


def normalizar_texto(valor):
    if pd.isna(valor):
        return pd.NA

    valor = str(valor).strip()
    valor = re.sub(r"\s+", " ", valor)

    if valor == "":
        return pd.NA

    return valor


def homologar_pais(valor):
    valor = normalizar_texto(valor)

    if pd.isna(valor):
        return pd.NA

    return COUNTRY_MAPPING.get(valor, valor)


def limpiar_columnas_texto(df):
    df_limpio = df.copy()
    columnas_texto = df_limpio.select_dtypes(include=["object", "string"]).columns

    for columna in columnas_texto:
        df_limpio[columna] = df_limpio[columna].apply(normalizar_texto)

    return df_limpio


def convertir_columnas_numericas(df, columnas):
    df_limpio = df.copy()

    for columna in columnas:
        if columna in df_limpio.columns:
            df_limpio[columna] = pd.to_numeric(df_limpio[columna], errors="coerce")

    return df_limpio


print("Funciones de limpieza creadas correctamente.")

Funciones de limpieza creadas correctamente.


### 3.2 Limpieza de fuentes

Se aplican funciones específicas para cada fuente de datos.  
El objetivo es obtener DataFrames con columnas normalizadas, tipos de datos correctos y campos preparados para la relación entre fuentes.


In [4]:
def limpiar_threats(df):
    df_limpio = df.copy()
    df_limpio.columns = [normalizar_nombre_columna(col) for col in df_limpio.columns]
    df_limpio = df_limpio.drop_duplicates().reset_index(drop=True)
    df_limpio = limpiar_columnas_texto(df_limpio)
    df_limpio["country"] = df_limpio["country"].apply(homologar_pais)

    columnas_numericas = [
        "year",
        "financial_loss_in_million",
        "number_of_affected_users",
        "incident_resolution_time_in_hours"
    ]

    df_limpio = convertir_columnas_numericas(df_limpio, columnas_numericas)
    df_limpio["year"] = df_limpio["year"].astype("Int64")

    return df_limpio


def limpiar_cyber_security(df):
    df_limpio = df.copy()
    df_limpio.columns = [normalizar_nombre_columna(col) for col in df_limpio.columns]
    df_limpio = df_limpio.drop_duplicates().reset_index(drop=True)
    df_limpio = limpiar_columnas_texto(df_limpio)
    df_limpio["country"] = df_limpio["country"].apply(homologar_pais)

    df_limpio["region"] = df_limpio["region"].replace({
        "Asia-Pasific": "Asia-Pacific"
    })

    columnas_numericas = ["cei", "gci", "ncsi", "ddl"]
    df_limpio = convertir_columnas_numericas(df_limpio, columnas_numericas)

    df_limpio["has_missing_security_index"] = df_limpio[columnas_numericas].isnull().any(axis=1)

    return df_limpio


def limpiar_internet_users(df):
    df_limpio = df.copy()
    df_limpio.columns = [normalizar_nombre_columna(col) for col in df_limpio.columns]

    if "country_or_area" in df_limpio.columns:
        df_limpio = df_limpio.rename(columns={"country_or_area": "country"})

    df_limpio = df_limpio.drop_duplicates().reset_index(drop=True)
    df_limpio = limpiar_columnas_texto(df_limpio)
    df_limpio["country"] = df_limpio["country"].apply(homologar_pais)

    columnas_numericas = [
        "internet_users",
        "population_2021",
        "year",
        "percentage_internet_users"
    ]

    df_limpio = convertir_columnas_numericas(df_limpio, columnas_numericas)
    df_limpio["year"] = df_limpio["year"].astype("Int64")
    df_limpio["has_missing_digital_data"] = df_limpio[columnas_numericas].isnull().any(axis=1)

    return df_limpio


df_threats_clean = limpiar_threats(df_threats)
df_cyber_security_clean = limpiar_cyber_security(df_cyber_security)
df_internet_users_clean = limpiar_internet_users(df_internet_users)

print("Fuentes limpiadas correctamente.")

Fuentes limpiadas correctamente.


### 3.3 Selección de columnas relevantes

Se seleccionan las columnas necesarias para el análisis y para la construcción posterior del modelo estrella.


In [5]:
columnas_threats = [
    "country",
    "year",
    "attack_type",
    "target_industry",
    "financial_loss_in_million",
    "number_of_affected_users",
    "attack_source",
    "security_vulnerability_type",
    "defense_mechanism_used",
    "incident_resolution_time_in_hours"
]

columnas_cyber_security = [
    "country",
    "region",
    "cei",
    "gci",
    "ncsi",
    "ddl",
    "has_missing_security_index"
]

columnas_internet_users = [
    "country",
    "subregion",
    "region",
    "internet_users",
    "population_2021",
    "year",
    "percentage_internet_users",
    "has_missing_digital_data"
]

df_threats_selected = df_threats_clean[columnas_threats].copy()
df_cyber_security_selected = df_cyber_security_clean[columnas_cyber_security].copy()
df_internet_users_selected = df_internet_users_clean[columnas_internet_users].copy()

print("Columnas relevantes seleccionadas correctamente.")

Columnas relevantes seleccionadas correctamente.


### 3.4 Funciones auxiliares de transformación

Estas funciones permiten crear claves limpias y categorizar variables numéricas para facilitar el análisis posterior.


In [6]:
def crear_clave_texto(valor):
    if pd.isna(valor):
        return pd.NA

    valor = str(valor).strip().lower()
    valor = unicodedata.normalize("NFKD", valor).encode("ascii", "ignore").decode("utf-8")
    valor = re.sub(r"[^a-z0-9]+", "_", valor)
    valor = re.sub(r"_+", "_", valor)
    valor = valor.strip("_")

    return valor


def categorizar_perdida_financiera(valor):
    if pd.isna(valor):
        return "Sin dato"
    if valor < 25:
        return "Baja"
    if valor < 50:
        return "Media"
    if valor < 75:
        return "Alta"
    return "Crítica"


def categorizar_usuarios_afectados(valor):
    if pd.isna(valor):
        return "Sin dato"
    if valor < 250_000:
        return "Bajo impacto"
    if valor < 500_000:
        return "Impacto medio"
    if valor < 750_000:
        return "Impacto alto"
    return "Impacto crítico"


def categorizar_tiempo_resolucion(valor):
    if pd.isna(valor):
        return "Sin dato"
    if valor <= 24:
        return "Resolución rápida"
    if valor <= 48:
        return "Resolución media"
    return "Resolución lenta"


def categorizar_indicador_0_100(valor):
    if pd.isna(valor):
        return "Sin dato"
    if valor < 40:
        return "Bajo"
    if valor < 70:
        return "Medio"
    if valor < 85:
        return "Alto"
    return "Muy alto"


def categorizar_penetracion_internet(valor):
    if pd.isna(valor):
        return "Sin dato"
    if valor < 40:
        return "Baja"
    if valor < 70:
        return "Media"
    if valor < 90:
        return "Alta"
    return "Muy alta"


print("Funciones auxiliares creadas correctamente.")

Funciones auxiliares creadas correctamente.


### 3.5 Generación de DataFrames transformados

Se crean los DataFrames transformados que servirán como base para el modelo estrella.


In [7]:
# Transformación del dataset de amenazas
df_threats_transformed = df_threats_selected.copy()

df_threats_transformed["country_key"] = df_threats_transformed["country"].apply(crear_clave_texto)

df_threats_transformed["country_year_key"] = (
    df_threats_transformed["country_key"]
    + "_"
    + df_threats_transformed["year"].astype(str)
)

df_threats_transformed["financial_loss_usd"] = (
    df_threats_transformed["financial_loss_in_million"] * 1_000_000
)

df_threats_transformed["incident_year_start_date"] = pd.to_datetime(
    df_threats_transformed["year"].astype(str) + "-01-01",
    errors="coerce"
)

df_threats_transformed["financial_loss_category"] = df_threats_transformed[
    "financial_loss_in_million"
].apply(categorizar_perdida_financiera)

df_threats_transformed["affected_users_category"] = df_threats_transformed[
    "number_of_affected_users"
].apply(categorizar_usuarios_afectados)

df_threats_transformed["resolution_time_category"] = df_threats_transformed[
    "incident_resolution_time_in_hours"
].apply(categorizar_tiempo_resolucion)


# Transformación del dataset de índices de ciberseguridad
df_cyber_security_transformed = df_cyber_security_selected.copy()

df_cyber_security_transformed["country_key"] = df_cyber_security_transformed[
    "country"
].apply(crear_clave_texto)

df_cyber_security_transformed["cei_percentage"] = df_cyber_security_transformed["cei"] * 100

indicadores_ciberseguridad = ["cei", "gci", "ncsi", "ddl"]

df_cyber_security_transformed["available_security_indicators"] = df_cyber_security_transformed[
    indicadores_ciberseguridad
].notnull().sum(axis=1)

df_cyber_security_transformed["gci_level"] = df_cyber_security_transformed[
    "gci"
].apply(categorizar_indicador_0_100)

df_cyber_security_transformed["ncsi_level"] = df_cyber_security_transformed[
    "ncsi"
].apply(categorizar_indicador_0_100)

df_cyber_security_transformed["ddl_level"] = df_cyber_security_transformed[
    "ddl"
].apply(categorizar_indicador_0_100)

df_cyber_security_transformed["has_complete_security_index"] = ~df_cyber_security_transformed[
    "has_missing_security_index"
]


# Transformación del dataset de usuarios de internet
df_internet_users_transformed = df_internet_users_selected.copy()

df_internet_users_transformed["country_key"] = df_internet_users_transformed[
    "country"
].apply(crear_clave_texto)

df_internet_users_transformed["country_year_key"] = (
    df_internet_users_transformed["country_key"]
    + "_"
    + df_internet_users_transformed["year"].astype(str)
)

df_internet_users_transformed["internet_users_millions"] = (
    df_internet_users_transformed["internet_users"] / 1_000_000
).round(2)

df_internet_users_transformed["population_millions"] = (
    df_internet_users_transformed["population_2021"] / 1_000_000
).round(2)

df_internet_users_transformed["internet_users_ratio"] = (
    df_internet_users_transformed["percentage_internet_users"] / 100
).round(4)

df_internet_users_transformed["internet_year_start_date"] = pd.to_datetime(
    df_internet_users_transformed["year"].astype(str) + "-01-01",
    errors="coerce"
)

df_internet_users_transformed["internet_penetration_level"] = df_internet_users_transformed[
    "percentage_internet_users"
].apply(categorizar_penetracion_internet)

print("DataFrames transformados generados correctamente.")

DataFrames transformados generados correctamente.


### 3.6 Creación de DataFrame enriquecido

Se crea un DataFrame enriquecido a partir de los incidentes de ciberseguridad, agregando indicadores de ciberseguridad y datos de usuarios de internet por país.


In [8]:
cyber_security_context = df_cyber_security_transformed[[
    "country",
    "region",
    "cei",
    "gci",
    "ncsi",
    "ddl",
    "cei_percentage",
    "gci_level",
    "ncsi_level",
    "ddl_level",
    "available_security_indicators",
    "has_complete_security_index"
]].rename(columns={
    "region": "security_region"
})

internet_users_context = df_internet_users_transformed[[
    "country",
    "subregion",
    "region",
    "year",
    "internet_users",
    "population_2021",
    "percentage_internet_users",
    "internet_users_millions",
    "population_millions",
    "internet_users_ratio",
    "internet_penetration_level"
]].rename(columns={
    "region": "internet_region",
    "year": "internet_data_year"
})

df_threats_enriched = (
    df_threats_transformed
    .merge(cyber_security_context, on="country", how="left")
    .merge(internet_users_context, on="country", how="left")
)

print("DataFrame enriquecido generado correctamente.")
print("Filas en df_threats_transformed:", df_threats_transformed.shape[0])
print("Filas en df_threats_enriched:", df_threats_enriched.shape[0])

DataFrame enriquecido generado correctamente.
Filas en df_threats_transformed: 3000
Filas en df_threats_enriched: 3000


In [9]:
dataframes_transformados = {
    "df_threats_transformed": df_threats_transformed,
    "df_cyber_security_transformed": df_cyber_security_transformed,
    "df_internet_users_transformed": df_internet_users_transformed,
    "df_threats_enriched": df_threats_enriched
}

resumen_transformados = pd.DataFrame([
    {
        "dataframe": nombre,
        "filas": df.shape[0],
        "columnas": df.shape[1],
        "valores_nulos": int(df.isnull().sum().sum()),
        "registros_duplicados": int(df.duplicated().sum())
    }
    for nombre, df in dataframes_transformados.items()
])

resumen_transformados

,dataframe,filas,columnas,valores_nulos,registros_duplicados
0,df_threats_transformed,3000,17,0,0
1,df_cyber_security_transformed,192,14,235,0
2,df_internet_users_transformed,215,15,0,0
3,df_threats_enriched,3000,38,0,0


## 4. Creación del modelo estrella

En esta sección se define el modelo estrella que será utilizado para estructurar los datos transformados en un formato analítico.

Un modelo estrella se compone de:

- **Tabla de hechos:** contiene los eventos principales del análisis y sus métricas.
- **Tablas de dimensiones:** contienen atributos descriptivos que permiten analizar los hechos desde diferentes perspectivas.

Para este proyecto, la tabla principal de hechos será `fact_cybersecurity_incidents`, ya que cada registro representa un incidente de ciberseguridad.

La granularidad de la tabla de hechos será:

> **Un registro por cada incidente de ciberseguridad reportado en el dataset Global Cybersecurity Threats.**


In [10]:
modelo_estrella = pd.DataFrame([
    {"tipo_tabla": "Dimensión", "nombre_tabla": "dim_country", "descripcion": "Países y atributos geográficos o contextuales."},
    {"tipo_tabla": "Dimensión", "nombre_tabla": "dim_year", "descripcion": "Años disponibles para el análisis temporal."},
    {"tipo_tabla": "Dimensión", "nombre_tabla": "dim_attack_type", "descripcion": "Tipos de ataques de ciberseguridad."},
    {"tipo_tabla": "Dimensión", "nombre_tabla": "dim_target_industry", "descripcion": "Industrias afectadas por incidentes."},
    {"tipo_tabla": "Dimensión", "nombre_tabla": "dim_attack_source", "descripcion": "Fuentes u orígenes de los ataques."},
    {"tipo_tabla": "Dimensión", "nombre_tabla": "dim_vulnerability", "descripcion": "Tipos de vulnerabilidades asociadas a incidentes."},
    {"tipo_tabla": "Dimensión", "nombre_tabla": "dim_defense_mechanism", "descripcion": "Mecanismos de defensa utilizados."},
    {"tipo_tabla": "Dimensión", "nombre_tabla": "dim_financial_loss_category", "descripcion": "Categorías de pérdida financiera."},
    {"tipo_tabla": "Dimensión", "nombre_tabla": "dim_affected_users_category", "descripcion": "Categorías de impacto según usuarios afectados."},
    {"tipo_tabla": "Dimensión", "nombre_tabla": "dim_resolution_time_category", "descripcion": "Categorías de tiempo de resolución."},
    {"tipo_tabla": "Hechos", "nombre_tabla": "fact_cybersecurity_incidents", "descripcion": "Incidentes de ciberseguridad y métricas principales."}
])

modelo_estrella

,tipo_tabla,nombre_tabla,descripcion
0,Dimensión,dim_country,Países y atributos geográficos o contextuales.
1,Dimensión,dim_year,Años disponibles para el análisis temporal.
2,Dimensión,dim_attack_type,Tipos de ataques de ciberseguridad.
3,Dimensión,dim_target_industry,Industrias afectadas por incidentes.
4,Dimensión,dim_attack_source,Fuentes u orígenes de los ataques.
5,Dimensión,dim_vulnerability,Tipos de vulnerabilidades asociadas a incidentes.
6,Dimensión,dim_defense_mechanism,Mecanismos de defensa utilizados.
7,Dimensión,dim_financial_loss_category,Categorías de pérdida financiera.
8,Dimensión,dim_affected_users_category,Categorías de impacto según usuarios afectados.
9,Dimensión,dim_resolution_time_category,Categorías de tiempo de resolución.


### 4.1 Estructura propuesta del modelo

Representación conceptual:

```text
                           dim_country
                               |
dim_attack_type -------- fact_cybersecurity_incidents -------- dim_year
                               |
dim_target_industry             |
                               |
dim_attack_source               |
                               |
dim_vulnerability               |
                               |
dim_defense_mechanism           |
                               |
dim_financial_loss_category     |
                               |
dim_affected_users_category     |
                               |
dim_resolution_time_category
```


## 5. Creación de dimensiones

En esta sección se crean las tablas de dimensiones del modelo estrella.

Cada dimensión contiene valores únicos de una categoría analítica y un identificador numérico secuencial.  
Estos identificadores serán usados posteriormente como claves foráneas en la tabla de hechos.


In [11]:
def crear_dimension(df, columnas_dimension, nombre_id, ordenar_por=None):
    dimension = df[columnas_dimension].drop_duplicates().copy()

    if ordenar_por is not None and ordenar_por in dimension.columns:
        dimension = dimension.sort_values(by=ordenar_por)
    else:
        dimension = dimension.sort_values(by=columnas_dimension)

    dimension = dimension.reset_index(drop=True)
    dimension.insert(0, nombre_id, range(1, len(dimension) + 1))

    return dimension


print("Función crear_dimension creada correctamente.")

Función crear_dimension creada correctamente.


### 5.1 Dimensión de países

La dimensión `dim_country` consolida los países presentes en las fuentes transformadas e incluye atributos de contexto geográfico y digital.


In [12]:
paises_unicos = sorted(
    set(df_threats_transformed["country"].dropna())
    .union(set(df_cyber_security_transformed["country"].dropna()))
    .union(set(df_internet_users_transformed["country"].dropna()))
)

dim_country = pd.DataFrame({
    "country": paises_unicos
})

dim_country["country_key"] = dim_country["country"].apply(crear_clave_texto)

country_security_context = (
    df_cyber_security_transformed[[
        "country",
        "region",
        "gci_level",
        "ncsi_level",
        "ddl_level",
        "has_complete_security_index"
    ]]
    .drop_duplicates(subset=["country"])
    .rename(columns={"region": "security_region"})
)

country_internet_context = (
    df_internet_users_transformed[[
        "country",
        "subregion",
        "region",
        "internet_penetration_level"
    ]]
    .drop_duplicates(subset=["country"])
    .rename(columns={"region": "internet_region"})
)

dim_country = (
    dim_country
    .merge(country_security_context, on="country", how="left")
    .merge(country_internet_context, on="country", how="left")
)

dim_country.insert(0, "country_id", range(1, len(dim_country) + 1))

dim_country.head()

,country_id,country,country_key,security_region,gci_level,ncsi_level,ddl_level,has_complete_security_index,subregion,internet_region,internet_penetration_level
0,1,Afghanistan,afghanistan,Asia-Pacific,Bajo,Bajo,Bajo,True,Southern Asia,Asia,Baja
1,2,Albania,albania,Europe,Medio,Medio,Medio,True,Southern Europe,Europe,Alta
2,3,Algeria,algeria,Africa,Bajo,Bajo,Medio,True,Northern Africa,Africa,Media
3,4,Andorra,andorra,Europe,Bajo,Sin dato,Sin dato,False,Southern Europe,Europe,Muy alta
4,5,Angola,angola,Africa,Bajo,Bajo,Bajo,False,Middle Africa,Africa,Baja


### 5.2 Dimensión de tiempo

La dimensión `dim_year` permite analizar los incidentes y datos digitales por año.


In [13]:
years_unicos = sorted(
    set(df_threats_transformed["year"].dropna().astype(int))
    .union(set(df_internet_users_transformed["year"].dropna().astype(int)))
)

dim_year = pd.DataFrame({
    "year": years_unicos
})

dim_year["year_start_date"] = pd.to_datetime(
    dim_year["year"].astype(str) + "-01-01",
    errors="coerce"
)

dim_year["year_end_date"] = pd.to_datetime(
    dim_year["year"].astype(str) + "-12-31",
    errors="coerce"
)

dim_year.insert(0, "year_id", range(1, len(dim_year) + 1))

dim_year

,year_id,year,year_start_date,year_end_date
0,1,2015,2015-01-01,2015-12-31
1,2,2016,2016-01-01,2016-12-31
2,3,2017,2017-01-01,2017-12-31
3,4,2018,2018-01-01,2018-12-31
4,5,2019,2019-01-01,2019-12-31
5,6,2020,2020-01-01,2020-12-31
6,7,2021,2021-01-01,2021-12-31
7,8,2022,2022-01-01,2022-12-31
8,9,2023,2023-01-01,2023-12-31
9,10,2024,2024-01-01,2024-12-31


### 5.3 Dimensiones descriptivas y categorías analíticas

Se crean las dimensiones relacionadas con los atributos principales de los incidentes y las categorías derivadas en la transformación.


In [14]:
dim_attack_type = crear_dimension(
    df_threats_transformed,
    ["attack_type"],
    "attack_type_id"
)

dim_target_industry = crear_dimension(
    df_threats_transformed,
    ["target_industry"],
    "target_industry_id"
)

dim_attack_source = crear_dimension(
    df_threats_transformed,
    ["attack_source"],
    "attack_source_id"
)

dim_vulnerability = crear_dimension(
    df_threats_transformed,
    ["security_vulnerability_type"],
    "vulnerability_id"
)

dim_defense_mechanism = crear_dimension(
    df_threats_transformed,
    ["defense_mechanism_used"],
    "defense_mechanism_id"
)

dim_financial_loss_category = crear_dimension(
    df_threats_transformed,
    ["financial_loss_category"],
    "financial_loss_category_id"
)

dim_affected_users_category = crear_dimension(
    df_threats_transformed,
    ["affected_users_category"],
    "affected_users_category_id"
)

dim_resolution_time_category = crear_dimension(
    df_threats_transformed,
    ["resolution_time_category"],
    "resolution_time_category_id"
)

print("Dimensiones creadas correctamente.")

Dimensiones creadas correctamente.


### 5.4 Resumen de dimensiones creadas

Se valida la cantidad de filas, columnas, duplicados y valores nulos de cada dimensión.


In [15]:
dimensiones = {
    "dim_country": dim_country,
    "dim_year": dim_year,
    "dim_attack_type": dim_attack_type,
    "dim_target_industry": dim_target_industry,
    "dim_attack_source": dim_attack_source,
    "dim_vulnerability": dim_vulnerability,
    "dim_defense_mechanism": dim_defense_mechanism,
    "dim_financial_loss_category": dim_financial_loss_category,
    "dim_affected_users_category": dim_affected_users_category,
    "dim_resolution_time_category": dim_resolution_time_category
}

resumen_dimensiones = pd.DataFrame([
    {
        "dimension": nombre,
        "filas": df.shape[0],
        "columnas": df.shape[1],
        "duplicados": int(df.duplicated().sum()),
        "valores_nulos": int(df.isnull().sum().sum())
    }
    for nombre, df in dimensiones.items()
])

resumen_dimensiones

,dimension,filas,columnas,duplicados,valores_nulos
0,dim_country,222,11,0,171
1,dim_year,10,4,0,0
2,dim_attack_type,6,2,0,0
3,dim_target_industry,7,2,0,0
4,dim_attack_source,4,2,0,0
5,dim_vulnerability,4,2,0,0
6,dim_defense_mechanism,5,2,0,0
7,dim_financial_loss_category,4,2,0,0
8,dim_affected_users_category,4,2,0,0
9,dim_resolution_time_category,3,2,0,0


## 6. Creación de tabla de hechos

En esta sección se crea la tabla de hechos `fact_cybersecurity_incidents`.

La tabla de hechos contiene las claves que conectan con las dimensiones y las métricas principales del análisis.

La granularidad de esta tabla es:

> **Un registro por cada incidente de ciberseguridad.**

Las métricas principales son:

- pérdida financiera en millones,
- pérdida financiera en dólares,
- número de usuarios afectados,
- tiempo de resolución en horas,
- indicadores de ciberseguridad del país,
- usuarios de internet,
- población,
- porcentaje de usuarios de internet.


In [16]:
fact_cybersecurity_incidents = df_threats_enriched.copy()

fact_cybersecurity_incidents = fact_cybersecurity_incidents.merge(
    dim_country[["country_id", "country"]],
    on="country",
    how="left"
)

fact_cybersecurity_incidents = fact_cybersecurity_incidents.merge(
    dim_year[["year_id", "year"]],
    on="year",
    how="left"
)

fact_cybersecurity_incidents = fact_cybersecurity_incidents.merge(
    dim_attack_type,
    on="attack_type",
    how="left"
)

fact_cybersecurity_incidents = fact_cybersecurity_incidents.merge(
    dim_target_industry,
    on="target_industry",
    how="left"
)

fact_cybersecurity_incidents = fact_cybersecurity_incidents.merge(
    dim_attack_source,
    on="attack_source",
    how="left"
)

fact_cybersecurity_incidents = fact_cybersecurity_incidents.merge(
    dim_vulnerability,
    on="security_vulnerability_type",
    how="left"
)

fact_cybersecurity_incidents = fact_cybersecurity_incidents.merge(
    dim_defense_mechanism,
    on="defense_mechanism_used",
    how="left"
)

fact_cybersecurity_incidents = fact_cybersecurity_incidents.merge(
    dim_financial_loss_category,
    on="financial_loss_category",
    how="left"
)

fact_cybersecurity_incidents = fact_cybersecurity_incidents.merge(
    dim_affected_users_category,
    on="affected_users_category",
    how="left"
)

fact_cybersecurity_incidents = fact_cybersecurity_incidents.merge(
    dim_resolution_time_category,
    on="resolution_time_category",
    how="left"
)

fact_cybersecurity_incidents.insert(
    0,
    "incident_fact_id",
    range(1, len(fact_cybersecurity_incidents) + 1)
)

print("Tabla de hechos creada correctamente.")
print("Filas:", fact_cybersecurity_incidents.shape[0])
print("Columnas:", fact_cybersecurity_incidents.shape[1])

Tabla de hechos creada correctamente.
Filas: 3000
Columnas: 49


### 6.1 Selección de columnas finales de la tabla de hechos

Se seleccionan únicamente las claves y métricas que deben quedar en la tabla de hechos, evitando duplicar información descriptiva que ya se encuentra en las dimensiones.


In [17]:
columnas_fact = [
    "incident_fact_id",
    "country_id",
    "year_id",
    "attack_type_id",
    "target_industry_id",
    "attack_source_id",
    "vulnerability_id",
    "defense_mechanism_id",
    "financial_loss_category_id",
    "affected_users_category_id",
    "resolution_time_category_id",
    "financial_loss_in_million",
    "financial_loss_usd",
    "number_of_affected_users",
    "incident_resolution_time_in_hours",
    "cei",
    "gci",
    "ncsi",
    "ddl",
    "internet_users",
    "population_2021",
    "percentage_internet_users"
]

fact_cybersecurity_incidents = fact_cybersecurity_incidents[columnas_fact].copy()

fact_cybersecurity_incidents.head()

,incident_fact_id,country_id,year_id,attack_type_id,target_industry_id,attack_source_id,vulnerability_id,defense_mechanism_id,financial_loss_category_id,affected_users_category_id,resolution_time_category_id,financial_loss_in_million,financial_loss_usd,number_of_affected_users,incident_resolution_time_in_hours,cei,gci,ncsi,ddl,internet_users,population_2021,percentage_internet_users
0,1,42,5,4,2,1,2,5,3,3,1,80.53,80530000.0,773169,63,0.483,92.53,51.95,62.41,1102140000,1425893465,77.3
1,2,42,5,5,6,1,2,4,1,4,1,62.19,62190000.0,295961,71,0.483,92.53,51.95,62.41,1102140000,1425893465,77.3
2,3,92,3,3,5,1,3,5,4,2,3,38.65,38650000.0,605895,20,0.597,97.50,59.74,40.02,881250000,1407563842,62.6
3,4,211,10,5,7,3,1,1,4,2,3,41.44,41440000.0,659320,7,0.207,99.54,89.61,79.96,65001016,67281039,96.6
4,5,76,4,3,5,2,1,5,1,3,1,74.41,74410000.0,810682,68,0.241,97.41,90.91,80.01,77794405,83408554,93.3


### 6.2 Validación de la tabla de hechos

Se valida que la tabla de hechos conserve todos los incidentes del dataset principal y que las claves de relación con las dimensiones no contengan valores nulos.


In [18]:
columnas_clave_fact = [
    "country_id",
    "year_id",
    "attack_type_id",
    "target_industry_id",
    "attack_source_id",
    "vulnerability_id",
    "defense_mechanism_id",
    "financial_loss_category_id",
    "affected_users_category_id",
    "resolution_time_category_id"
]

validacion_fact = pd.DataFrame([
    {
        "validacion": "La tabla de hechos conserva todos los incidentes",
        "resultado": fact_cybersecurity_incidents.shape[0] == df_threats_transformed.shape[0],
        "detalle": f"{fact_cybersecurity_incidents.shape[0]} registros en hechos vs {df_threats_transformed.shape[0]} incidentes transformados"
    },
    {
        "validacion": "incident_fact_id es único",
        "resultado": fact_cybersecurity_incidents["incident_fact_id"].is_unique,
        "detalle": f"{fact_cybersecurity_incidents['incident_fact_id'].nunique()} identificadores únicos"
    },
    {
        "validacion": "No existen claves nulas en la tabla de hechos",
        "resultado": fact_cybersecurity_incidents[columnas_clave_fact].isnull().sum().sum() == 0,
        "detalle": f"{int(fact_cybersecurity_incidents[columnas_clave_fact].isnull().sum().sum())} claves nulas encontradas"
    },
    {
        "validacion": "No existen duplicados completos en la tabla de hechos",
        "resultado": fact_cybersecurity_incidents.duplicated().sum() == 0,
        "detalle": f"{int(fact_cybersecurity_incidents.duplicated().sum())} duplicados completos"
    }
])

validacion_fact

,validacion,resultado,detalle
0,La tabla de hechos conserva todos los incidentes,True,3000 registros en hechos vs 3000 incidentes tr...
1,incident_fact_id es único,True,3000 identificadores únicos
2,No existen claves nulas en la tabla de hechos,True,0 claves nulas encontradas
3,No existen duplicados completos en la tabla de...,True,0 duplicados completos


### 6.3 Resumen final del modelo estrella

Se consolida un resumen con todas las dimensiones y la tabla de hechos creada.


In [19]:
tablas_modelo_estrella = {
    **dimensiones,
    "fact_cybersecurity_incidents": fact_cybersecurity_incidents
}

resumen_modelo_estrella = pd.DataFrame([
    {
        "tabla": nombre,
        "tipo": "Hechos" if nombre.startswith("fact_") else "Dimensión",
        "filas": df.shape[0],
        "columnas": df.shape[1],
        "duplicados": int(df.duplicated().sum()),
        "valores_nulos": int(df.isnull().sum().sum())
    }
    for nombre, df in tablas_modelo_estrella.items()
])

resumen_modelo_estrella

,tabla,tipo,filas,columnas,duplicados,valores_nulos
0,dim_country,Dimensión,222,11,0,171
1,dim_year,Dimensión,10,4,0,0
2,dim_attack_type,Dimensión,6,2,0,0
3,dim_target_industry,Dimensión,7,2,0,0
4,dim_attack_source,Dimensión,4,2,0,0
5,dim_vulnerability,Dimensión,4,2,0,0
6,dim_defense_mechanism,Dimensión,5,2,0,0
7,dim_financial_loss_category,Dimensión,4,2,0,0
8,dim_affected_users_category,Dimensión,4,2,0,0
9,dim_resolution_time_category,Dimensión,3,2,0,0


In [20]:
# Verificación final de secciones principales del notebook
secciones_principales = [
    "## 1. Introducción del proyecto",
    "## 2. Importación de librerías",
    "## 3. Carga de DataFrames transformados",
    "## 4. Creación del modelo estrella",
    "## 5. Creación de dimensiones",
    "## 6. Creación de tabla de hechos"
]

for seccion in secciones_principales:
    print(seccion)

## 1. Introducción del proyecto
## 2. Importación de librerías
## 3. Carga de DataFrames transformados
## 4. Creación del modelo estrella
## 5. Creación de dimensiones
## 6. Creación de tabla de hechos
